In [2]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print("PyTorch Version:", torch.__version__)

CUDA Available: True
GPU Device: NVIDIA L4
PyTorch Version: 2.6.0+cu124


## Text-only forward pass demo

A minimal nanochat-like transformer is implemented below (PyTorch). It demonstrates a text-only forward pass producing logits for each token.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Load and run the real nanochat (text-only forward pass)

The cells below will attempt to install/clone `nanochat`, import its `GPT` model and tokenizer, instantiate a small model, and run a text-only forward pass (logits output). Run them in order.

In [4]:
# Install or make nanochat importable. This uses pip to fetch the repo; if that fails we clone the repo and add it to sys.path.
import sys, subprocess, os
def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg], check=False)
try:
    import nanochat
    print('nanochat already importable at', getattr(nanochat, '__file__', None))
except Exception as e:
    print('Installing nanochat from GitHub...')
    pip_install('git+https://github.com/karpathy/nanochat.git')
    if not os.path.exists('nanochat'):
        subprocess.run(['git','clone','https://github.com/karpathy/nanochat.git'], check=False)
    sys.path.insert(0, os.path.abspath('nanochat'))
    import importlib
    importlib.invalidate_caches()
    import nanochat
    print('Imported nanochat from', getattr(nanochat, '__file__', None))

nanochat already importable at None


In [9]:
# Text-only forward pass using nanochat's GPT implementation (random init).
import torch
import nanochat
from nanochat.gpt import GPT, GPTConfig
# Try to use the HuggingFace-compatible tokenizer wrapper if available
try:
    from nanochat.tokenizer import HuggingFaceTokenizer
except Exception:
    HuggingFaceTokenizer = None
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
# Build or load a tokenizer (fallback to simple char tokenizer if HF tokenizer unavailable)
if HuggingFaceTokenizer is not None:
    try:
        tokenizer = HuggingFaceTokenizer.from_pretrained('gpt2')
        vocab_size = tokenizer.get_vocab_size()
        print('Loaded HF tokenizer. Vocab size:', vocab_size)
        def encode_text(text):
            ids = tokenizer.encode(text)
            if isinstance(ids, list) and len(ids) and isinstance(ids[0], list):
                return ids[0]
            return ids
    except Exception as e:
        print('HF tokenizer load failed, falling back to simple tokenizer:', e)
        tokenizer = None
else:
    tokenizer = None
if tokenizer is None:
    # Fallback simple char tokenizer
    print('Using fallback char tokenizer')
    vocab = {'<pad>':0, '<unk>':1}
    def encode_text(text):
        return [vocab.setdefault(ch, len(vocab)) for ch in text]
    vocab_size = len(vocab)
# Instantiate a small GPT config + model (randomly initialized). Adjust sizes for speed/VRAM.
cfg = GPTConfig(sequence_len=128, vocab_size=vocab_size, n_layer=2, n_head=2, n_kv_head=2, n_embd=128)
model = GPT(cfg).to(device)
# Some builds provide init_weights(); call if available
try:
    model.init_weights()
except Exception:
    pass
model.eval()
text = 'Hello world'
ids = encode_text(text)
input_ids = torch.tensor([ids], dtype=torch.long, device=device)
with torch.no_grad():
    logits = model(input_ids)
print('logits.shape =', getattr(logits, 'shape', None))

device: cuda
Using fallback char tokenizer
Padding vocab_size from 2 to 64 for efficiency
logits.shape = torch.Size([1, 11, 2])


If the cells above error, ensure the kernel has internet access and that required packages (e.g., `tokenizers`) can be installed. The earlier toy model cells can be used as a fallback for quick prototyping.

In [6]:
import os, sys, importlib

candidates = [
    os.path.abspath("nanochat"),
    os.path.abspath("multimodal-nanochat/nanochat"),
    os.path.abspath("/home/as7629/multimodal-nanochat/nanochat"),
]
for p in candidates:
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, "nanochat", "__init__.py")):
        if p not in sys.path:
            sys.path.insert(0, p)
        print("added to sys.path:", p)
        break
else:
    raise FileNotFoundError("nanochat package not found; clone the repo into the workspace or adjust the path")

importlib.invalidate_caches()
import nanochat.common
print("Imported:", nanochat.common.__file__)

added to sys.path: /home/as7629/multimodal-nanochat/nanochat


ModuleNotFoundError: No module named 'nanochat.common'

In [5]:
from nanochat.common import get_base_dir
import os, glob
base = get_base_dir()
print('base dir:', base)
print('contents:', os.listdir(base))
print('found checkpoints:', glob.glob(os.path.join(base, '**', 'model_*.pt'), recursive=True))

ModuleNotFoundError: No module named 'nanochat.common'

In [7]:
import sys, os, importlib, importlib.util, pkgutil
print('sys.executable:', sys.executable)
print('python version:', sys.version.splitlines()[0])
print('--- sys.path ---')
for p in sys.path:
    print(' ', p)
repo_parent = '/home/as7629/multimodal-nanochat/nanochat'
print('repo_parent exists:', os.path.isdir(repo_parent))
print('repo_parent listing:', os.listdir(repo_parent) if os.path.isdir(repo_parent) else 'N/A')
inner = os.path.join(repo_parent, 'nanochat')
print('inner package exists:', os.path.isdir(inner))
print('inner listing (first 20):', os.listdir(inner)[:20] if os.path.isdir(inner) else 'N/A')
print('--- importlib.find_spec ---')
print('nanochat spec:', importlib.util.find_spec('nanochat'))
print('nanochat.common spec:', importlib.util.find_spec('nanochat.common'))
print('--- pkgutil.iter_modules(repo_parent) ---')
print(list(pkgutil.iter_modules([repo_parent])))

sys.executable: /home/as7629/nanochat_baseline/bin/python
python version: 3.10.17 | packaged by conda-forge | (main, Apr 10 2025, 22:19:12) [GCC 13.3.0]
--- sys.path ---
  /home/as7629/multimodal-nanochat/nanochat
  /opt/conda/lib/python310.zip
  /opt/conda/lib/python3.10
  /opt/conda/lib/python3.10/lib-dynload
  
  /home/as7629/nanochat_baseline/lib/python3.10/site-packages
  /opt/conda/lib/python3.10/site-packages
repo_parent exists: True
repo_parent listing: ['README.md', 'scripts', '.python-version', 'runs', 'tests', '.gitignore', 'uv.lock', '.git', 'tasks', '.claude', 'LICENSE', 'dev', 'nanochat', 'pyproject.toml']
inner package exists: True
inner listing (first 20): ['report.py', 'gpt.py', 'logo.svg', 'core_eval.py', 'tokenizer.py', 'execution.py', 'checkpoint_manager.py', 'flash_attention.py', 'ui.html', '__init__.py', 'engine.py', 'optim.py', '__pycache__', 'common.py', 'fp8.py', 'loss_eval.py', 'dataset.py', 'dataloader.py']
--- importlib.find_spec ---
nanochat spec: ModuleSpe

In [8]:
import sys, os, importlib, importlib.util, types

candidates = [
    os.path.abspath("nanochat"),
    os.path.abspath("multimodal-nanochat/nanochat"),
    os.path.abspath("/home/as7629/multimodal-nanochat/nanochat"),
    os.path.abspath("/home/as7629/multimodal-nanochat"),
]
repo_root = None
inner = None
for p in candidates:
    q = os.path.join(p, "nanochat")
    if os.path.isdir(q) and os.path.isfile(os.path.join(q, "__init__.py")):
        repo_root, inner = p, q
        break
if repo_root is None:
    for p in candidates:
        if os.path.isdir(p) and os.path.isfile(os.path.join(p, "__init__.py")):
            inner = p
            repo_root = os.path.dirname(p)
            break
if repo_root is None:
    raise FileNotFoundError("nanochat package not found; adjust the path or clone the repo into the workspace")

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
importlib.invalidate_caches()

# Ensure a package object exists and has __path__ pointing to the inner package dir
if 'nanochat' not in sys.modules:
    pkg = types.ModuleType('nanochat')
    pkg.__path__ = [inner]
    sys.modules['nanochat'] = pkg
else:
    mod = sys.modules['nanochat']
    if not hasattr(mod, '__path__'):
        mod.__path__ = [inner]
    elif inner not in mod.__path__:
        mod.__path__.insert(0, inner)

def load_mod(mod_name, filepath):
    spec = importlib.util.spec_from_file_location(mod_name, filepath)
    module = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = module
    spec.loader.exec_module(module)
    return module

# Try normal import first, otherwise fall back to file-loading
try:
    import nanochat.common
    print("Imported nanochat.common normally:", nanochat.common.__file__)
except Exception:
    print("Normal import failed — falling back to file loads")
    base = inner
    common = load_mod("nanochat.common", os.path.join(base, "common.py"))
    gpt = load_mod("nanochat.gpt", os.path.join(base, "gpt.py"))
    tokenizer = load_mod("nanochat.tokenizer", os.path.join(base, "tokenizer.py"))
    print("Loaded:", common, gpt, tokenizer)

print("nanochat.__path__ =", sys.modules['nanochat'].__path__)


AttributeError: '_NamespacePath' object has no attribute 'insert'